In [1]:
!pip install yfinance
!pip install newsapi-python
!pip install transformers
!pip install scikit-learn

  Obtaining dependency information for yfinance from https://files.pythonhosted.org/packages/dc/bc/e46ed5dfb88c6f7af0f641ffb6227d32f484ea989a2987a52a9c35d17aa9/yfinance-1.3.0-py2.py3-none-any.whl.metadata
  Using cached yfinance-1.3.0-py2.py3-none-any.whl.metadata (6.1 kB)
  Obtaining dependency information for curl_cffi>=0.15 from https://files.pythonhosted.org/packages/d8/8c/2abf99a38d6340d66cf0557e0c750ef3f8883dfc5d450087e01c85861343/curl_cffi-0.15.0-cp310-abi3-win_amd64.whl.metadata
  Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl.metadata (18 kB)
  Obtaining dependency information for cffi>=2.0.0 from https://files.pythonhosted.org/packages/ae/8f/dc5531155e7070361eb1b7e4c1a9d896d0cb21c49f807a6c03fd63fc877e/cffi-2.0.0-cp311-cp311-win_amd64.whl.metadata
  Using cached cffi-2.0.0-cp311-cp311-win_amd64.whl.metadata (2.6 kB)
  Obtaining dependency information for certifi>=2024.2.2 from https://files.pythonhosted.org/packages/22/30/7cd8fdcdfbc5b869528b079bfb76dcdf6056b1a2097a662

  Obtaining dependency information for newsapi-python from https://files.pythonhosted.org/packages/74/47/e3b099102f0c826d37841d2266e19f1568dcf58ba86e4c6948e2a124f91d/newsapi_python-0.2.7-py2.py3-none-any.whl.metadata
  Using cached newsapi_python-0.2.7-py2.py3-none-any.whl.metadata (1.2 kB)


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import pickle

In [3]:
data = yf.download("^NSEI", start="2018-01-01")

# Fix multi-index issue
data.columns = data.columns.get_level_values(0)

data.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Date,,,,,
2018-01-02,10442.200195,10495.200195,10404.650391,10477.549805,153400
2018-01-03,10443.200195,10503.599609,10429.549805,10482.650391,167300
2018-01-04,10504.799805,10513.000000,10441.450195,10469.400391,174900
2018-01-05,10558.849609,10566.099609,10520.099609,10534.250000,180900
2018-01-08,10623.599609,10631.200195,10588.549805,10591.700195,169000


In [4]:
data = yf.download("^NSEI", start="2018-01-01")

# Fix multi-index issue
data.columns = data.columns.get_level_values(0)

data.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Date,,,,,
2018-01-02,10442.200195,10495.200195,10404.650391,10477.549805,153400
2018-01-03,10443.200195,10503.599609,10429.549805,10482.650391,167300
2018-01-04,10504.799805,10513.000000,10441.450195,10469.400391,174900
2018-01-05,10558.849609,10566.099609,10520.099609,10534.250000,180900
2018-01-08,10623.599609,10631.200195,10588.549805,10591.700195,169000


In [5]:
data['Target'] = data['Close'].shift(-1)

data['Movement'] = (
    data['Target'] > data['Close']
).astype(int)

In [6]:
data['Returns'] = data['Close'].pct_change()

In [7]:
# Moving averages
data['MA10'] = data['Close'].rolling(10).mean()
data['MA50'] = data['Close'].rolling(50).mean()
data['MA200'] = data['Close'].rolling(200).mean()

# Volatility
data['Volatility'] = data['Returns'].rolling(10).std()

# Momentum
data['Momentum'] = data['Close'] - data['Close'].shift(10)

In [8]:
delta = data['Close'].diff()

gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()

rs = gain / loss
data['RSI'] = 100 - (100 / (1 + rs))

In [9]:
data['Return_Lag1'] = data['Returns'].shift(1)
data['Return_Lag2'] = data['Returns'].shift(2)
data['Return_Lag3'] = data['Returns'].shift(3)

In [10]:
data['Geo_Index'] = 0.0   # float type

In [11]:
data['Return_Lag1'] = data['Returns'].shift(1)
data['Return_Lag2'] = data['Returns'].shift(2)
data['Return_Lag3'] = data['Returns'].shift(3)

In [12]:
data.dropna(inplace=True)

In [13]:
print(data.columns)

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'Target', 'Movement',
       'Returns', 'MA10', 'MA50', 'MA200', 'Volatility', 'Momentum', 'RSI',
       'Return_Lag1', 'Return_Lag2', 'Return_Lag3', 'Geo_Index'],
      dtype='object', name='Price')


In [14]:
features = [
    'Returns',
    'Return_Lag1',
    'Return_Lag2',
    'Return_Lag3',
    'MA10',
    'MA50',
    'MA200',
    'Volatility',
    'Momentum',
    'RSI',
    'Geo_Index'
]

In [16]:
X = data[features]
y = data['Movement']

split = int(len(data)*0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [17]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=12, n_estimators=300, random_state=42)

In [18]:
from sklearn.metrics import accuracy_score

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Model Accuracy:", accuracy)

Model Accuracy: 0.49326145552560646


In [20]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance)

Price
Returns        0.116350
Volatility     0.103643
Return_Lag1    0.102647
Return_Lag3    0.102091
RSI            0.099840
MA200          0.098065
Momentum       0.097469
Return_Lag2    0.095710
MA10           0.092641
MA50           0.091544
Geo_Index      0.000000
dtype: float64


In [21]:
probs = model.predict_proba(X_test)

confidence = probs[:,1]


In [22]:
from sklearn.metrics import accuracy_score

strong = confidence > 0.6

filtered_accuracy = accuracy_score(
    y_test[strong],
    predictions[strong]
)

print("High Confidence Accuracy:", filtered_accuracy)

High Confidence Accuracy: 0.42105263157894735


In [23]:
def market_signal(X):
    prob = model.predict_proba(X)[0][1]

    if prob > 0.7:
        return "🔥 Strong Bullish", prob
    elif prob > 0.6:
        return "📈 Bullish", prob
    elif prob > 0.5:
        return "📈 Weak Bullish", prob
    else:
        return "📉 Bearish", prob

In [25]:
latest = X.iloc[-1:].copy()

market_signal(latest)

('📈 Weak Bullish', 0.5932646886413043)

In [26]:
import pickle

pickle.dump(model, open("market_model.pkl","wb"))

In [29]:
!pip install newsapi-python transformers

In [30]:
from newsapi import NewsApiClient

newsapi = NewsApiClient(api_key="ececdf9eec604b70b624b6d1d75f87f4")

articles = newsapi.get_everything(
    q="war OR sanctions OR oil OR election OR geopolitics",
    language="en",
    sort_by="publishedAt"
)

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis")

scores = []

for a in articles['articles'][:20]:
    res = sentiment(a['title'])[0]
    score = res['score'] if res['label']=="POSITIVE" else -res['score']
    scores.append(score)

geo_index = sum(scores)/len(scores)

print("Geopolitical Index:", geo_index)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Geopolitical Index: -0.18106483221054076


In [ ]:
latest = X.iloc[-1:].copy()

latest.loc[:, 'Geo_Index'] = float(geo_index)

market_signal(latest)

('📉 Bearish', np.float64(0.44564968784516573))